<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 01. Valores Faltantes — ¿Qué hacemos con los huecos en la libreta?
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/05%20-%20Data%20Preparation/Para%20Dummies/01_Valores_Faltantes_Data_Preparation_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Los valores faltantes (`NaN`) son el problema más común en datos reales. En este cuaderno aprenderás:

1. Qué es un valor faltante y por qué aparece.
2. Los **3 tipos** de datos faltantes (y por qué importa la diferencia).
3. Cómo **detectar** cuántos hay y dónde están.
4. Las **2 estrategias** principales: borrar o rellenar (imputar).
5. Cómo **rellenar** con la media, la mediana o el valor más frecuente.

---
## 1. La analogía del libro de calificaciones 📒

Imagina la libreta de notas de un colegio. Algunos campos están en blanco porque:

- El estudiante **faltó ese día** y no presentó el examen → *falta completamente al azar*.
- Los estudiantes con **nota muy baja** prefirieron no reportarla → *la ausencia depende del valor mismo*.
- El sistema informático tuvo un **error al grabar** ese día → *falta por causa externa*.

Cada causa requiere una solución diferente. En Python, ese campo en blanco se representa como `NaN` (Not a Number).

| Tipo técnico | Causa | Ejemplo |
|---|---|---|
| **MCAR** (falta completamente al azar) | Error aleatorio, encuesta no completada | Un sensor que falló un día puntual |
| **MAR** (falta según otro dato) | La ausencia depende de otra columna | Edad no reportada solo por los mayores de 65 |
| **MNAR** (falta según su propio valor) | El valor falta porque es extremo | Los pacientes graves no se presentaron al control |

In [ ]:
import os, urllib.parse, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def load_dataset(filename, module_name="05 - Data Preparation"):
    candidates = [f"data/{filename}", f"../{module_name}/data/{filename}", f"{module_name}/data/{filename}", filename]
    for path in candidates:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    target_path = f"data/{filename}"
    url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{urllib.parse.quote(module_name)}/data/{urllib.parse.quote(filename)}"
    urllib.request.urlretrieve(url, target_path)
    print(f"✅ Dataset '{filename}' descargado.")
    return target_path

df = pd.read_csv(load_dataset('hepatitis.csv'), na_values='?')
print(f"\n📊 Dataset cargado: {df.shape[0]} pacientes × {df.shape[1]} columnas")
df.head()

---
## 2. Detectar los huecos — ¿Dónde está el problema? 🔍

In [ ]:
# Conteo de nulos por columna
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
resumen_nulos = pd.DataFrame({'Nulos': nulos, '% del total': nulos_pct}).sort_values('% del total', ascending=False)
resumen_nulos = resumen_nulos[resumen_nulos['Nulos'] > 0]

print("🕳️ Columnas con valores faltantes:")
print(resumen_nulos.to_string())
print(f"\n📌 Total: {nulos.sum()} celdas vacías en {(nulos > 0).sum()} columnas")

In [ ]:
# Mapa de calor de nulos — ver el patrón visualmente
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False,
            cmap=['#f0fdf4', '#ef4444'])  # verde=dato, rojo=nulo
plt.title('🗺️ Mapa de valores faltantes\n(rojo = celda vacía, verde = dato presente)',
          fontweight='bold', fontsize=12)
plt.xlabel('Columnas')
plt.tight_layout()
plt.show()
print("\n💡 Cada fila es un paciente. Las zonas rojas son los datos que faltan.")

---
## 3. Estrategia 1 — Eliminar filas o columnas con nulos 🗑️

**¿Cuándo eliminar?**
- Cuando una **columna** tiene más del 40-50% de nulos → eliminar la columna completa.
- Cuando una **fila** tiene la mayoría de columnas vacías → eliminar esa fila.
- Cuando el dataset es muy grande y los nulos son pocos.

**¿Cuándo NO eliminar?**
- Cuando el dataset es pequeño y perder filas significaría perder información valiosa.
- Cuando la ausencia del dato tiene significado (ej. el paciente no se presentó porque se curó).

In [ ]:
print(f"Tamaño original: {df.shape}")

# Eliminar filas que tengan AL MENOS UN nulo
df_sin_nulos = df.dropna()
print(f"Después de eliminar filas con cualquier nulo: {df_sin_nulos.shape}")
print(f"  → Perdimos {len(df) - len(df_sin_nulos)} filas ({(len(df)-len(df_sin_nulos))/len(df)*100:.1f}% de los datos)")

# Eliminar columnas que tengan más del 30% de nulos
umbral = 0.30
cols_muchos_nulos = nulos_pct[nulos_pct > umbral * 100].index.tolist()
df_cols_limpias = df.drop(columns=cols_muchos_nulos)
print(f"\nColumnas eliminadas por >30% de nulos: {cols_muchos_nulos}")
print(f"Tamaño tras eliminar columnas: {df_cols_limpias.shape}")

print("\n⚠️  En este dataset perderíamos demasiados datos. Mejor imputar.")

---
## 4. Estrategia 2 — Rellenar (Imputar) los huecos 🩹

**Imputar** = rellenar los valores faltantes con un valor calculado. Las opciones más comunes son:

| Estrategia | Qué hace | Úsala cuando... |
|---|---|---|
| **Media** | Rellena con el promedio de la columna | La columna es numérica y sin muchos outliers |
| **Mediana** | Rellena con el valor del medio | La columna tiene outliers (ej. salarios) |
| **Moda** | Rellena con el valor más frecuente | La columna es categórica (texto) |
| **KNN** | Rellena según los vecinos más parecidos | Cuando hay relación entre columnas |

> 💡 **Analogía:** Imputar con la media es como poner el promedio de la clase cuando un estudiante faltó al examen. No es el valor real, pero es la mejor estimación sin más información.

In [ ]:
from sklearn.impute import SimpleImputer

# Separar columnas numéricas y categóricas
numericas = df.select_dtypes(include='number').columns.tolist()
categoricas = df.select_dtypes(exclude='number').columns.tolist()

print(f"Columnas numéricas ({len(numericas)}): {numericas[:5]}...")
print(f"Columnas categóricas ({len(categoricas)}): {categoricas}")

# Imputar numéricos con la MEDIANA
imp_mediana = SimpleImputer(strategy='median')
df_num_imp = pd.DataFrame(
    imp_mediana.fit_transform(df[numericas]),
    columns=numericas
)

# Imputar categóricos con la MODA (most_frequent)
imp_moda = SimpleImputer(strategy='most_frequent')
df_cat_imp = pd.DataFrame(
    imp_moda.fit_transform(df[categoricas]),
    columns=categoricas
)

# Combinar de nuevo
df_imputado = pd.concat([df_num_imp, df_cat_imp], axis=1)

print(f"\n✅ Nulos ANTES de imputar: {df.isnull().sum().sum()}")
print(f"✅ Nulos DESPUÉS de imputar: {df_imputado.isnull().sum().sum()}")

---
## 5. Comparación visual: antes vs después de imputar 📊

In [ ]:
col_ejemplo = 'albumin'  # columna con nulos en hepatitis
if col_ejemplo in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    df[col_ejemplo].dropna().hist(bins=15, ax=axes[0], color='#ef4444', alpha=0.8, edgecolor='white')
    axes[0].set_title(f'{col_ejemplo} — ANTES de imputar\n(sin los NaN)', fontweight='bold')
    axes[0].set_xlabel('Valor')

    df_imputado[col_ejemplo].hist(bins=15, ax=axes[1], color='#10b981', alpha=0.8, edgecolor='white')
    axes[1].set_title(f'{col_ejemplo} — DESPUÉS de imputar\n(con mediana agregada)', fontweight='bold')
    axes[1].set_xlabel('Valor')

    plt.suptitle('Efecto de la imputación en la distribución', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("\n💡 Verás un 'pico' en la mediana en el gráfico derecho — ahí están los valores imputados.")

---
## 6. Árbol de decisión — ¿Borrar o rellenar? 🌳

```
  ¿Cuántos nulos tiene la columna?
            │
    ┌───────┴───────┐
    ▼               ▼
 > 50% nulos    < 50% nulos
    │               │
 Eliminar la    ¿Es numérica o categórica?
  columna            │
              ┌──────┴──────┐
              ▼             ▼
          Numérica      Categórica
              │             │
      ¿Hay outliers?    Imputar con
        Sí → Mediana      la Moda
        No → Media
```

> 🚀 **Siguiente paso:** Ve al cuaderno `02_Escalado_Caracteristicas_Data_Preparation_Dummies.ipynb` para aprender a poner todas las variables en la misma escala.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>